In [ ]:
# Colab setup. In a local clone you can comment this out.
!pip install -q git+https://github.com/2forts/qcirclab_repo.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from math import sqrt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

from qcirclab import Circuit
import qcirclab.gates as qg

from qcirclab.viz import (
    pretty_state,
    counts_from_probs,
    sample_counts_from_statevector,
    counts_to_probvec,
    plot_counts,
)

from qcirclab.noise import (
    state_to_density,
    probabilities_from_density,
    apply_kraus_to_qubit,
    amplitude_damping_kraus,
    phase_damping_kraus,
    depolarizing_channel,
    expectation,
    fidelity_pure_density,
)

from qcirclab.operators import (
    circuit_unitary,
    inverse_circuit,
)

## Subsection **6.2.1 Decoherence and relaxation (T1, T2)**

In [ ]:
# Prepare |+>, apply amplitude damping and phase damping, and sample.
gamma_1 = 0.3
Gamma_phi = 0.2

qc = Circuit(1)
qc.h(0)
psi = qc.statevector()
rho = state_to_density(psi)

rho_noisy = apply_kraus_to_qubit(rho, amplitude_damping_kraus(gamma_1), 0, 1)
rho_noisy = apply_kraus_to_qubit(rho_noisy, phase_damping_kraus(Gamma_phi), 0, 1)

counts = counts_from_probs(probabilities_from_density(rho_noisy), shots=10000, seed=1)
print("Noisy measurement counts:", counts)
print("Noisy density matrix:\n", rho_noisy)
plot_counts(counts, title="Amplitude + phase damping on |+>")

## Subsection **6.2.2 Gate imperfections and calibration errors**

In [ ]:
# Ideal Bell circuit, followed by a simple depolarizing model on both qubits.
qc = Circuit(2)
qc.h(0)
qc.cx(0, 1)
psi_ideal = qc.statevector()
rho_ideal = state_to_density(psi_ideal)

p1 = 0.001
p2_effective = 0.01
rho_noisy = rho_ideal.copy()
# A toy two-qubit depolarizing proxy: apply local depolarization to both qubits.
rho_noisy = depolarizing_channel(rho_noisy, p2_effective, qubit=0, n_qubits=2)
rho_noisy = depolarizing_channel(rho_noisy, p2_effective, qubit=1, n_qubits=2)

ideal_counts = counts_from_probs(probabilities_from_density(rho_ideal), shots=10000, seed=2)
noisy_counts = counts_from_probs(probabilities_from_density(rho_noisy), shots=10000, seed=3)
print("Ideal counts:", ideal_counts)
print("Noisy counts:", noisy_counts)
plot_counts(ideal_counts, title="Ideal Bell state")
plot_counts(noisy_counts, title="Bell state with depolarizing noise")

## Subsection **6.2.3 Measurement and readout errors**

In [ ]:
# Readout error as a classical confusion matrix.
n = 1
qc = Circuit(n)
qc.h(0)
state = qc.statevector()
ideal_probs = np.abs(state)**2

p01, p10 = 0.06, 0.04  # P(report 1|true 0), P(report 0|true 1)
M = np.array([[1 - p01, p10],
              [p01,     1 - p10]])  # columns = true states

noisy_probs = M @ ideal_probs
p_mitigated = np.linalg.pinv(M) @ noisy_probs
p_mitigated = np.maximum(p_mitigated, 0)
p_mitigated /= p_mitigated.sum()

print("Ideal probabilities:    ", ideal_probs)
print("Noisy probabilities:    ", noisy_probs)
print("Mitigated probabilities:", p_mitigated)

## Subsection **6.3.2 Quantum channels and Kraus representation**

In [ ]:
gamma = 0.25

Ks_amp = amplitude_damping_kraus(gamma)
Ks_phase = phase_damping_kraus(gamma)

def check_kraus_completeness(kraus_ops):
    total = np.zeros((2, 2), dtype=complex)
    for K in kraus_ops:
        total += K.conj().T @ K
    return total

print("Amplitude-damping Kraus operators:")
for K in Ks_amp:
    print(K)

print("Completeness check:")
print(check_kraus_completeness(Ks_amp))

print("Phase-damping Kraus operators:")
for K in Ks_phase:
    print(K)

print("Completeness check:")
print(check_kraus_completeness(Ks_phase))

## Subsection **6.3.3 Lindblad master equation**

In [ ]:
sigma_minus = np.array([[0, 1], [0, 0]], dtype=complex)
sigma_plus = sigma_minus.conj().T
gamma = 1.0

def lindblad_rhs_complex(t, rho_vec):
    rho = rho_vec.reshape((2, 2))
    L = gamma * (sigma_minus @ rho @ sigma_plus - 0.5 * (sigma_plus @ sigma_minus @ rho + rho @ sigma_plus @ sigma_minus))
    return L.reshape(4)

def rhs_real(t, y):
    y_complex = y[:4] + 1j * y[4:]
    dydt_complex = lindblad_rhs_complex(t, y_complex)
    return np.concatenate([dydt_complex.real, dydt_complex.imag])

rho0 = np.array([[0, 0], [0, 1]], dtype=complex)
y0 = np.concatenate([rho0.reshape(4).real, rho0.reshape(4).imag])
t_eval = np.linspace(0, 5, 200)
sol = solve_ivp(rhs_real, (0, 5), y0, t_eval=t_eval, rtol=1e-10, atol=1e-12)
rho_t = (sol.y[:4] + 1j * sol.y[4:]).T
pop_excited = np.array([np.real(rt.reshape((2,2))[1,1]) for rt in rho_t])

plt.figure()
plt.plot(t_eval, pop_excited, label="numerical")
plt.plot(t_eval, np.exp(-gamma*t_eval), "--", label="analytic exp(-γt)")
plt.xlabel("time")
plt.ylabel("excited-state population")
plt.legend()
plt.show()

## Subsection **6.4.1 Structure of toy noise models**

In [ ]:
# qcirclab does not use backend-attached noise models, but the same information
# can be represented explicitly as a lightweight configuration dictionary.
toy_noise_model = {
    "single_qubit": {
        0: {"depolarizing": 0.0008, "amplitude_damping": 0.03, "phase_damping": 0.02},
        1: {"depolarizing": 0.0015, "phase_damping": 0.02},
    },
    "two_qubit": {
        "cx": {"depolarizing_proxy": 0.012}
    },
    "readout": {
        0: {"p01": 0.03, "p10": 0.02},
        1: {"p01": 0.04, "p10": 0.03},
    }
}

toy_noise_model

## Subsection **6.4.2 Simulating ideal vs. noisy circuits**

In [ ]:
base = Circuit(2)
base.h(0)
base.cx(0, 1)

psi_ideal = base.statevector()
rho_noisy = state_to_density(psi_ideal)

# Apply single-qubit noise from the explicit configuration.
for q, params in toy_noise_model["single_qubit"].items():
    if "amplitude_damping" in params:
        rho_noisy = apply_kraus_to_qubit(
            rho_noisy,
            amplitude_damping_kraus(params["amplitude_damping"]),
            q, 2)

    if "phase_damping" in params:
        rho_noisy = apply_kraus_to_qubit(
            rho_noisy,
            phase_damping_kraus(params["phase_damping"]),
            q, 2)

    if "depolarizing" in params:
        rho_noisy = depolarizing_channel(
            rho_noisy, params["depolarizing"], q, 2)

# Apply a simple effective two-qubit gate noise proxy.
p_cx = toy_noise_model["two_qubit"]["cx"]["depolarizing_proxy"]
for q in [0, 1]:
    rho_noisy = depolarizing_channel(rho_noisy, p_cx, q, 2)

F = fidelity_pure_density(psi_ideal, rho_noisy)

print("Ideal Bell state:", pretty_state(psi_ideal))
print("Fidelity with noisy density matrix:", F)
print("Noisy probabilities:", probabilities_from_density(rho_noisy))

## Subsection **6.5.1 Quantum process tomography**

In [ ]:
# Minimal process characterization: estimate the Pauli transfer matrix
# of a one-qubit depolarizing channel using exact simulation.
paulis = [qg.I2, qg.X, qg.Y, qg.Z]
labels = ["I", "X", "Y", "Z"]
p = 0.10

def channel(rho):
    return depolarizing_channel(rho, p, qubit=0, n_qubits=1)

# Input operators for Pauli-transfer construction.
R = np.zeros((4, 4))
for j, Pj in enumerate(paulis):
    # Treat Pj as an operator input and apply the channel
    # linearly.
    E_Pj = (1-p)*Pj + (p/3)*(qg.X@Pj@qg.X
                             + qg.Y@Pj@qg.Y
                             + qg.Z@Pj@qg.Z)
    for i, Pi in enumerate(paulis):
        R[i, j] = 0.5 * np.trace(Pi @ E_Pj).real

print("Pauli transfer matrix labels:", labels)
print(R)

## Subsection **6.5.2 Randomized benchmarking**

In [ ]:
# Toy RB with random H/S words. We compute the exact inverse matrix of each word,
# append it as a unitary, and then apply a simple depolarizing survival model.
np.random.seed(123)
seq_lengths = [1, 2, 4, 8, 16, 32, 64]
n_trials = 50
p_gate = 0.002

single_gates = [("h", qg.H), ("s", qg.S), ("x", qg.X)]

def random_word_unitary(m):
    U = np.eye(2, dtype=complex)
    for _ in range(m):
        _, G = single_gates[np.random.randint(len(single_gates))]
        U = G @ U
    return U

survival = []
for m in seq_lengths:
    vals = []
    for _ in range(n_trials):
        U = random_word_unitary(m)
        qc = Circuit(1)
        qc.unitary(U, [0], name="word")
        qc.unitary(U.conj().T, [0], name="inv")
        # Ideal sequence returns |0>. Add a toy exponential decay model.
        vals.append(0.5 + 0.5 * (1 - p_gate)**m)
    survival.append(np.mean(vals))

def rb_decay(m, A, alpha, B):
    return A * (alpha ** m) + B

popt, _ = curve_fit(rb_decay, np.array(seq_lengths), np.array(survival), p0=(0.5, 0.99, 0.5))
plt.figure()
plt.plot(seq_lengths, survival, "o", label="toy data")
plt.plot(seq_lengths, rb_decay(np.array(seq_lengths), *popt), "-", label=f"fit alpha={popt[1]:.4f}")
plt.xlabel("sequence length")
plt.ylabel("survival probability")
plt.legend()
plt.show()

## Subsection **6.5.3 Noise diagnostics and empirical characterization**

In [ ]:
# Ramsey-like dephasing experiment: prepare |+>, apply phase damping for many idle steps,
# rotate back with H, and estimate <Z>.
gamma = 0.02
idle_steps = np.arange(0, 100, 5)
exp_values = []

for steps in idle_steps:
    qc = Circuit(1)
    qc.h(0)
    rho = state_to_density(qc.statevector())
    for _ in range(steps):
        rho = apply_kraus_to_qubit(rho, phase_damping_kraus(gamma), 0, 1)
    # Measure X by applying H before Z measurement: <X> = Tr(X rho)
    exp_values.append(expectation(rho, qg.X))


def exp_decay(x, A, n2, B):
    return A * np.exp(-x / n2) + B

popt, _ = curve_fit(exp_decay, idle_steps, exp_values, p0=(1.0, 30.0, 0.0))
plt.figure()
plt.plot(idle_steps, exp_values, "o", label="simulation data")
plt.plot(idle_steps, exp_decay(idle_steps, *popt), "-", label=f"fit: n2={popt[1]:.2f}")
plt.xlabel("number of idle steps")
plt.ylabel("coherence signal <X>")
plt.legend()
plt.show()

## Subsection **6.6.1 Measurement error mitigation**

In [ ]:
p01, p10 = 0.08, 0.05
M = np.array([[1 - p01, p10], [p01, 1 - p10]])
qc = Circuit(1)
qc.h(0)
p_ideal = np.abs(qc.statevector())**2
p_noisy = M @ p_ideal
p_mitigated = np.linalg.pinv(M) @ p_noisy
p_mitigated = np.maximum(p_mitigated, 0)
p_mitigated /= p_mitigated.sum()

print("Ideal probabilities:     ", p_ideal)
print("Noisy probabilities:     ", p_noisy)
print("Mitigated probabilities: ", p_mitigated)

## Subsection **6.6.2 Zero-noise extrapolation**

In [ ]:
# Zero-noise extrapolation with a toy depolarizing-noise scale.
base = Circuit(1)
base.h(0)
base.ry(np.pi / 3, 0)
base.h(0)

U_base = circuit_unitary(base)
initial = np.array([1, 0], dtype=complex)

def folded_unitary(qc, scale):
    """Return U, U U† U, U U† U U† U, ... for odd scale."""
    U = circuit_unitary(qc)

    if scale == 1:
        return U

    out = np.eye(U.shape[0], dtype=complex)
    for i in range(scale):
        out = U @ out
        if i < scale - 1:
            out = U.conj().T @ out
    return out

p_noise = 0.01
scales = np.array([1, 3, 5])
expectations = []

for scale in scales:
    # The folded unitary has the same ideal action as U_base.
    U = folded_unitary(base, int(scale))
    psi = U @ initial
    rho = state_to_density(psi)

    # In this toy model, larger scale means more noise exposure.
    for _ in range(int(scale)):
        rho = depolarizing_channel(rho, p_noise, 0, 1)

    expectations.append(expectation(rho, qg.Z))

# Fit <Z>(scale) = a * scale + b and evaluate at scale = 0.
coeff = np.polyfit(scales, expectations, deg=1)
zne_estimate = np.polyval(coeff, 0)

ideal_exp = expectation(
    state_to_density(U_base @ initial),
    qg.Z
)

print("Noisy expectations:", dict(zip(scales, expectations)))
print("ZNE estimate:", zne_estimate)
print("Ideal expectation:", ideal_exp)

plt.figure()
plt.plot(scales, expectations, "o", label="noisy estimates")
plt.plot(
    [0, *scales],
    np.polyval(coeff, [0, *scales]),
    "--",
    label="linear extrapolation"
)
plt.xlabel("noise scale")
plt.ylabel("<Z>")
plt.legend()
plt.show()

## Subsection **6.6.3 Probabilistic error cancellation**



In [ ]:
# First-order probabilistic error cancellation for a one-qubit
# depolarizing channel.
p = 0.05
pauli_mats = [qg.I2, qg.X, qg.Y, qg.Z]
pauli_labels = ["I", "X", "Y", "Z"]
O = qg.X

qc = Circuit(1)
qc.h(0)

rho_ideal = state_to_density(qc.statevector())
rho_noisy = depolarizing_channel(rho_ideal, p, 0, 1)

exp_ideal = expectation(rho_ideal, O)
exp_noisy = expectation(rho_noisy, O)

weights = np.array([1 + 4*p/3, -p/3, -p/3, -p/3])
probs = np.abs(weights) / np.sum(np.abs(weights))
signs = np.sign(weights)
gamma = np.sum(np.abs(weights))

rng = np.random.default_rng(123)
N = 10000
samples = []

for _ in range(N):
    j = rng.choice(len(pauli_mats), p=probs)
    P = pauli_mats[j]
    rho_eff = P @ rho_noisy @ P
    val = signs[j] * gamma * expectation(rho_eff, O)
    samples.append(val)

exp_pec = np.mean(samples)

print("Ideal <X>: ", exp_ideal)
print("Noisy <X>: ", exp_noisy)
print("PEC <X>:   ", exp_pec)
print("PEC overhead gamma:", gamma)

## Subsection **6.6.4 Symmetry verification**

In [ ]:
# Symmetry verification example: the Bell state should live in
# the even-parity sector {00, 11}.
bell = Circuit(2)
bell.h(0).cx(0, 1)

rho = state_to_density(bell.statevector())

# Add local noise that can populate odd-parity outcomes.
p = 0.03
rho_noisy = depolarizing_channel(rho, p, qubit=0, n_qubits=2)
rho_noisy = depolarizing_channel(rho_noisy, p, qubit=1, n_qubits=2)

probs_noisy = probabilities_from_density(rho_noisy)
labels = ["00", "01", "10", "11"]

# Keep only bitstrings with even parity.
valid = np.array([True, False, False, True])
probs_verified = probs_noisy.copy()
probs_verified[~valid] = 0.0
probs_verified /= probs_verified.sum()

ZZ = np.kron(qg.Z, qg.Z)

print("Noisy probabilities:")
for label, prob in zip(labels, probs_noisy):
    print(label, prob)

print("Verified probabilities:")
for label, prob in zip(labels, probs_verified):
    print(label, prob)

print("<ZZ> before verification:", expectation(rho_noisy, ZZ))
print("Probability kept:", probs_noisy[valid].sum())